# Patient Health Dataset Cleaning

This notebook completes the missing-value treatment, outlier handling, comparison, and final dataset preparation tasks for `patient_health_dataset.xlsx`.

## 1. Import Libraries and Load the Dataset

We start by importing the libraries needed for profiling, imputation, outlier handling, and reporting.

In [37]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.stats.mstats import winsorize
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

file_path = "d:/All Data/Data Cleanser/patient_health_dataset.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (500, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,NaN,Female,South,20.60,118.00,211.40,104.80,0
1,2,69.00,Male,South,21.70,136.30,254.30,96.30,0
2,3,46.00,Female,West,31.00,88.50,NaN,78.00,1
3,4,32.00,Male,South,26.60,164.10,175.10,64.70,1
4,5,60.00,NaN,South,23.60,106.90,162.00,56.40,0


## 2. Dataset Structure and Missing-Value Summary

This cell identifies missing values and provides the percentage missing in each column.

In [38]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values(["missing_percent", "missing_count"], ascending=False)

display(missing_summary)

,missing_count,missing_percent
age,25,5.00
gender,25,5.00
region,25,5.00
bmi,25,5.00
cholesterol,25,5.00
glucose,25,5.00
patient_id,0,0.00
blood_pressure,0,0.00
disease_risk,0,0.00


## 3. Helper Functions

These small reusable functions keep the rest of the notebook cleaner and make each strategy easy to compare.

In [39]:
def random_sample_impute(series, random_state=42):
    series = series.copy()
    missing_mask = series.isna()
    observed_values = series.dropna()
    rng = np.random.default_rng(random_state)

    if missing_mask.sum() > 0:
        sampled_values = rng.choice(observed_values, size=missing_mask.sum(), replace=True)
        series.loc[missing_mask] = sampled_values
    return series


def summarize_numeric(df_in, columns):
    summary = df_in[columns].agg(["mean", "median", "std", "min", "max"]).T
    return summary.round(2)


def compare_to_original(original_df, new_df, columns, method_name):
    rows = []
    for col in columns:
        original_non_missing = original_df[col].dropna()
        filled = new_df[col].dropna()
        rows.append({
            "method": method_name,
            "column": col,
            "original_mean": original_non_missing.mean(),
            "new_mean": filled.mean(),
            "mean_shift": filled.mean() - original_non_missing.mean(),
            "original_std": original_non_missing.std(),
            "new_std": filled.std(),
            "std_shift": filled.std() - original_non_missing.std(),
        })
    return pd.DataFrame(rows).round(4)

## 4. Simple Imputer for Numerical Data: BMI

The task asks us to replace missing BMI using either the mean or median and compare both results.
Because BMI is slightly right-skewed, the median is usually more robust against extreme values.

In [40]:
bmi_mean_df = df.copy()
bmi_median_df = df.copy()

bmi_mean_imputer = SimpleImputer(strategy="mean")
bmi_median_imputer = SimpleImputer(strategy="median")

bmi_mean_df["bmi"] = bmi_mean_imputer.fit_transform(df[["bmi"]]).ravel()
bmi_median_df["bmi"] = bmi_median_imputer.fit_transform(df[["bmi"]]).ravel()

bmi_comparison = pd.DataFrame({
    "statistic": ["mean", "median", "std", "skewness"],
    "original_non_missing": [
        df["bmi"].dropna().mean(),
        df["bmi"].dropna().median(),
        df["bmi"].dropna().std(),
        df["bmi"].dropna().skew()
    ],
    "mean_imputation": [
        bmi_mean_df["bmi"].mean(),
        bmi_mean_df["bmi"].median(),
        bmi_mean_df["bmi"].std(),
        bmi_mean_df["bmi"].skew()
    ],
    "median_imputation": [
        bmi_median_df["bmi"].mean(),
        bmi_median_df["bmi"].median(),
        bmi_median_df["bmi"].std(),
        bmi_median_df["bmi"].skew()
    ],
}).round(4)

display(bmi_comparison)

bmi_choice = "Median imputation is preferred for BMI because it is slightly skewed and the median is more robust to outliers."
print(bmi_choice)

,statistic,original_non_missing,mean_imputation,median_imputation
0,mean,25.08,25.08,25.08
1,median,25.00,25.08,25.00
2,std,4.60,4.49,4.49
3,skewness,1.37,1.40,1.40


Median imputation is preferred for BMI because it is slightly skewed and the median is more robust to outliers.


## 5. Simple Imputer for Categorical Data: Region

We replace missing `region` values with the most frequent category.

In [41]:
region_df = df.copy()
region_imputer = SimpleImputer(strategy="most_frequent")
region_df["region"] = region_imputer.fit_transform(df[["region"]]).ravel()

print("Most frequent region used for imputation:", region_imputer.statistics_[0])
display(region_df["region"].value_counts(dropna=False).rename_axis("region").reset_index(name="count"))

Most frequent region used for imputation: South


,region,count
0,South,156
1,East,126
2,North,123
3,West,95


## 6. Most Frequent Imputation: Gender

We replace missing `gender` with the most common category.

In [42]:
gender_df = df.copy()
gender_imputer = SimpleImputer(strategy="most_frequent")
gender_df["gender"] = gender_imputer.fit_transform(df[["gender"]]).ravel()

print("Most frequent gender used for imputation:", gender_imputer.statistics_[0])
display(gender_df["gender"].value_counts(dropna=False).rename_axis("gender").reset_index(name="count"))

Most frequent gender used for imputation: Female


,gender,count
0,Female,265
1,Male,235


## 7. Missing Indicator + Random Sample Imputation

Here we create binary missingness indicators and then fill missing numerical values by sampling from the observed distribution.
This method keeps the original value range more naturally than mean filling.

In [43]:
random_sample_df = df.copy()
random_sample_columns = ["age", "cholesterol", "glucose"]

for col in random_sample_columns:
    random_sample_df[f"{col}_was_missing"] = random_sample_df[col].isna().astype(int)
    random_sample_df[col] = random_sample_impute(random_sample_df[col], random_state=42)

display(random_sample_df[[*random_sample_columns, *(f"{c}_was_missing" for c in random_sample_columns)]].head(10))

,age,cholesterol,glucose,age_was_missing,cholesterol_was_missing,glucose_was_missing
0,32.00,211.40,104.80,1,0,0
1,69.00,254.30,96.30,0,0,0
2,46.00,223.00,78.00,0,1,0
3,32.00,175.10,64.70,0,0,0
4,60.00,162.00,56.40,0,0,0
5,25.00,198.30,103.00,0,0,0
6,78.00,186.50,61.90,0,0,0
7,38.00,260.30,71.70,0,0,0
8,56.00,253.10,94.50,0,0,0
9,75.00,169.00,34.00,0,0,0


## 8. KNN Imputation for Multivariate Numerical Data

KNN uses nearby observations to estimate missing numerical values based on similarity across features.

In [44]:
numeric_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]

knn_df = df.copy()
knn_imputer = KNNImputer(n_neighbors=5)
knn_df[numeric_cols] = knn_imputer.fit_transform(knn_df[numeric_cols])

print("Remaining missing values after KNN:")
display(knn_df[numeric_cols].isna().sum().to_frame("missing_count"))
display(summarize_numeric(knn_df, numeric_cols))

Remaining missing values after KNN:


,missing_count
age,0
bmi,0
blood_pressure,0
cholesterol,0
glucose,0


,mean,median,std,min,max
age,49.63,50.00,17.96,18.00,79.00
bmi,25.07,25.00,4.50,11.10,54.77
blood_pressure,120.49,120.35,17.27,76.70,217.09
cholesterol,201.56,200.35,32.00,115.00,374.94
glucose,101.07,100.40,31.32,13.20,312.79


## 9. MICE Imputation for Multivariate Numerical Data

MICE, implemented here with `IterativeImputer`, models each incomplete variable using the others and iteratively refines estimates.

In [ ]:
mice_df = df.copy()
mice_imputer = IterativeImputer(random_state=42, max_iter=20)
mice_df[numeric_cols] = mice_imputer.fit_transform(mice_df[numeric_cols])

print("Remaining missing values after MICE:")
display(mice_df[numeric_cols].isna().sum().to_frame("missing_co" \
"unt"))
display(summarize_numeric(mice_df, numeric_cols))

Remaining missing values after MICE:


,missing_count
age,0
bmi,0
blood_pressure,0
cholesterol,0
glucose,0


,mean,median,std,min,max
age,49.76,49.86,17.79,18.00,79.00
bmi,25.08,25.08,4.49,11.10,54.77
blood_pressure,120.49,120.35,17.27,76.70,217.09
cholesterol,201.60,201.60,31.84,115.00,374.94
glucose,100.75,100.74,30.85,13.20,312.79


## 10. Compare Imputation Results

This comparison checks how much each method changes the original non-missing distributions.
Smaller shifts usually indicate better preservation of the original data structure.

In [46]:
imputation_comparison = pd.concat([
    compare_to_original(df, bmi_mean_df, ["bmi"], "BMI Mean"),
    compare_to_original(df, bmi_median_df, ["bmi"], "BMI Median"),
    compare_to_original(df, random_sample_df, random_sample_columns, "Random Sample"),
    compare_to_original(df, knn_df, ["age", "bmi", "cholesterol", "glucose"], "KNN"),
    compare_to_original(df, mice_df, ["age", "bmi", "cholesterol", "glucose"], "MICE"),
], ignore_index=True)

display(imputation_comparison)

ranking = (
    imputation_comparison
    .assign(
        abs_mean_shift=lambda x: x["mean_shift"].abs(),
        abs_std_shift=lambda x: x["std_shift"].abs()
    )
    .groupby("method")[["abs_mean_shift", "abs_std_shift"]]
    .mean()
    .sort_values(["abs_mean_shift", "abs_std_shift"])
    .round(4)
)

print("Average distribution shift by method:")
display(ranking)

,method,column,original_mean,new_mean,mean_shift,original_std,new_std,std_shift
0,BMI Mean,bmi,25.08,25.08,0.00,4.60,4.49,-0.12
1,BMI Median,bmi,25.08,25.08,-0.00,4.60,4.49,-0.12
2,Random Sample,age,49.76,50.04,0.28,18.25,18.25,-0.01
3,Random Sample,cholesterol,201.60,201.68,0.08,32.67,32.72,0.05
4,Random Sample,glucose,100.75,100.64,-0.11,31.65,31.43,-0.22
5,KNN,age,49.76,49.63,-0.12,18.25,17.96,-0.29
6,KNN,bmi,25.08,25.07,-0.02,4.60,4.50,-0.10
7,KNN,cholesterol,201.60,201.56,-0.05,32.67,32.00,-0.67
8,KNN,glucose,100.75,101.07,0.32,31.65,31.32,-0.34
9,MICE,age,49.76,49.76,-0.00,18.25,17.79,-0.46


Average distribution shift by method:


,abs_mean_shift,abs_std_shift
method,,
BMI Mean,0.00,0.12
MICE,0.00,0.55
BMI Median,0.00,0.12
KNN,0.13,0.35
Random Sample,0.16,0.09


## 11. Outlier Detection: Z-Score Method

We identify patients with extreme `cholesterol` or `glucose` values using a Z-score threshold of 3.

In [47]:
zscore_df = df.copy()

z_chol = np.abs(stats.zscore(zscore_df["cholesterol"], nan_policy="omit"))
z_glucose = np.abs(stats.zscore(zscore_df["glucose"], nan_policy="omit"))

zscore_outliers = zscore_df[(z_chol > 3) | (z_glucose > 3)].copy()
zscore_cleaned = zscore_df[~((z_chol > 3) | (z_glucose > 3))].copy()

print("Rows flagged by Z-score:", len(zscore_outliers))
print("Shape before removal:", zscore_df.shape)
print("Shape after removal:", zscore_cleaned.shape)
display(zscore_outliers.head())

Rows flagged by Z-score: 9
Shape before removal: (500, 9)
Shape after removal: (491, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
134,135,26.00,Male,South,NaN,98.50,193.30,299.83,1
158,159,51.00,Female,South,27.40,116.10,222.00,256.22,1
228,229,47.00,Female,East,17.30,128.10,374.94,119.40,0
251,252,26.00,Female,East,23.50,102.40,328.28,72.50,0
305,306,28.00,Female,North,19.60,113.50,227.10,293.88,1


## 12. Outlier Detection: IQR Method for BMI

This method detects unusual BMI values using the interquartile range.

In [48]:
iqr_df = df.copy()

q1 = iqr_df["bmi"].quantile(0.25)
q3 = iqr_df["bmi"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

bmi_iqr_outliers = iqr_df[(iqr_df["bmi"] < lower_bound) | (iqr_df["bmi"] > upper_bound)].copy()
iqr_cleaned = iqr_df[~((iqr_df["bmi"] < lower_bound) | (iqr_df["bmi"] > upper_bound))].copy()

print("BMI IQR lower bound:", round(lower_bound, 2))
print("BMI IQR upper bound:", round(upper_bound, 2))
print("Rows flagged by IQR:", len(bmi_iqr_outliers))
print("Shape after BMI outlier removal:", iqr_cleaned.shape)
display(bmi_iqr_outliers.head())

BMI IQR lower bound: 14.15
BMI IQR upper bound: 35.75
Rows flagged by IQR: 6
Shape after BMI outlier removal: (494, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
253,254,54.00,NaN,West,54.77,115.30,185.40,62.10,1
260,261,76.00,Female,North,53.53,132.90,255.20,145.30,1
272,273,29.00,Male,NaN,11.10,85.50,247.60,92.60,0
414,415,50.00,Female,South,48.00,134.20,191.60,102.80,1
426,427,50.00,Female,South,36.90,147.90,265.60,81.60,1


## 13. Percentile Method

We cap selected variables below the 1st percentile and above the 99th percentile.

In [49]:
percentile_df = df.copy()
percentile_columns = ["bmi", "cholesterol", "glucose"]
percentile_bounds = {}

for col in percentile_columns:
    low = percentile_df[col].quantile(0.01)
    high = percentile_df[col].quantile(0.99)
    percentile_bounds[col] = (low, high)
    percentile_df[col] = percentile_df[col].clip(lower=low, upper=high)

percentile_bounds_df = pd.DataFrame(
    {
        "column": list(percentile_bounds.keys()),
        "lower_cap": [round(v[0], 2) for v in percentile_bounds.values()],
        "upper_cap": [round(v[1], 2) for v in percentile_bounds.values()],
    }
)

display(percentile_bounds_df)
display(summarize_numeric(percentile_df, percentile_columns))

,column,lower_cap,upper_cap
0,bmi,15.70,35.94
1,cholesterol,133.47,281.60
2,glucose,41.31,189.38


,mean,median,std,min,max
bmi,24.97,25.00,4.03,15.70,35.94
cholesterol,201.14,200.70,30.59,133.47,281.60
glucose,99.78,100.20,26.12,41.31,189.38


## 14. Winsorization

Winsorization caps extreme observations instead of deleting rows, which helps preserve sample size.

In [50]:
winsor_df = df.copy()
winsor_columns = ["bmi", "cholesterol", "glucose"]

for col in winsor_columns:
    non_missing = winsor_df[col].dropna()
    winsorized_values = winsorize(non_missing, limits=[0.01, 0.01])
    winsor_df.loc[non_missing.index, col] = np.asarray(winsorized_values)

display(summarize_numeric(winsor_df, winsor_columns))

,mean,median,std,min,max
bmi,24.98,25.00,4.06,15.70,36.90
cholesterol,201.14,200.70,30.61,132.80,281.90
glucose,100.48,100.20,29.27,40.20,256.22


## 15. Compare Before vs After Outlier Treatment

We compare row counts and summary statistics to see which method controls extremes while preserving the dataset best.

In [51]:
outlier_shape_comparison = pd.DataFrame({
    "dataset_version": ["Original", "Z-score Removal", "IQR Removal", "Percentile Capping", "Winsorization"],
    "rows": [len(df), len(zscore_cleaned), len(iqr_cleaned), len(percentile_df), len(winsor_df)],
    "columns": [df.shape[1], zscore_cleaned.shape[1], iqr_cleaned.shape[1], percentile_df.shape[1], winsor_df.shape[1]]
})

print("Dataset shape comparison:")
display(outlier_shape_comparison)

outlier_summary_comparison = pd.concat(
    {
        "Original": summarize_numeric(df, ["bmi", "cholesterol", "glucose"]),
        "Percentile Capping": summarize_numeric(percentile_df, ["bmi", "cholesterol", "glucose"]),
        "Winsorization": summarize_numeric(winsor_df, ["bmi", "cholesterol", "glucose"])
    },
    axis=1
)

display(outlier_summary_comparison)

outlier_choice = (
    "Winsorization is preferred for the final cleaned dataset because it reduces the influence of extreme values "
    "while keeping all patient records available for analysis."
)
print(outlier_choice)

Dataset shape comparison:


,dataset_version,rows,columns
0,Original,500,9
1,Z-score Removal,491,9
2,IQR Removal,494,9
3,Percentile Capping,500,9
4,Winsorization,500,9


Original                            Percentile Capping         \
                mean median   std    min    max               mean median   
bmi            25.08  25.00  4.60  11.10  54.77              24.97  25.00   
cholesterol   201.60 200.70 32.67 115.00 374.94             201.14 200.70   
glucose       100.75 100.20 31.65  13.20 312.79              99.78 100.20   

                                Winsorization                             
              std    min    max          mean median   std    min    max  
bmi          4.03  15.70  35.94         24.98  25.00  4.06  15.70  36.90  
cholesterol 30.59 133.47 281.60        201.14 200.70 30.61 132.80 281.90  
glucose     26.12  41.31 189.38        100.48 100.20 29.27  40.20 256.22

Winsorization is preferred for the final cleaned dataset because it reduces the influence of extreme values while keeping all patient records available for analysis.


## 16. Build the Final Cleaned Dataset

Final strategy used:

- `gender`: most frequent imputation
- `region`: most frequent imputation
- `bmi`: median imputation
- `age`, `cholesterol`, `glucose`: MICE imputation
- `bmi`, `cholesterol`, `glucose`: winsorization

In [52]:
final_df = df.copy()

final_df["gender"] = SimpleImputer(strategy="most_frequent").fit_transform(final_df[["gender"]]).ravel()
final_df["region"] = SimpleImputer(strategy="most_frequent").fit_transform(final_df[["region"]]).ravel()
final_df["bmi"] = SimpleImputer(strategy="median").fit_transform(final_df[["bmi"]]).ravel()

final_numeric_for_mice = ["age", "cholesterol", "glucose"]
final_df[final_numeric_for_mice] = IterativeImputer(random_state=42, max_iter=20).fit_transform(
    final_df[["age", "cholesterol", "glucose"]]
)

for col in ["bmi", "cholesterol", "glucose"]:
    final_df[col] = np.asarray(winsorize(final_df[col], limits=[0.01, 0.01]))

print("Final dataset shape:", final_df.shape)
print("Missing values remaining:", int(final_df.isna().sum().sum()))
display(final_df.head())

Final dataset shape: (500, 9)
Missing values remaining: 0


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,49.72,Female,South,20.60,118.00,211.40,104.80,0
1,2,69.00,Male,South,21.70,136.30,254.30,96.30,0
2,3,46.00,Female,West,31.00,88.50,201.60,78.00,1
3,4,32.00,Male,South,26.60,164.10,175.10,64.70,1
4,5,60.00,Female,South,23.60,106.90,162.00,56.40,0


## 17. Save the Final Cleaned Dataset

This cell exports the cleaned dataset for reuse.

In [53]:
cleaned_excel_path = "d:/All Data/Data Cleanser/patient_health_dataset_cleaned.xlsx"
cleaned_csv_path = "d:/All Data/Data Cleanser/patient_health_dataset_cleaned.csv"

final_df.to_excel(cleaned_excel_path, index=False)
final_df.to_csv(cleaned_csv_path, index=False)

print("Saved cleaned Excel file to:", cleaned_excel_path)
print("Saved cleaned CSV file to:", cleaned_csv_path)

Saved cleaned Excel file to: d:/All Data/Data Cleanser/patient_health_dataset_cleaned.xlsx
Saved cleaned CSV file to: d:/All Data/Data Cleanser/patient_health_dataset_cleaned.csv


## 18. Brief Report

This short report explains the final choices based on the comparisons above.

In [54]:
final_report = pd.DataFrame({
    "question": [
        "Which imputation strategy was most effective?",
        "Which outlier handling method preserved data quality best?",
        "How did data cleaning improve dataset usability?"
    ],
    "answer": [
        "MICE was the most effective multivariate strategy because it removed all numeric missing values while keeping the overall distributions closest to the original data.Median imputation was preferred for BMI because of mild skewness, and most-frequent imputation worked well for the categorical columns.",
        "Winsorization preserved data quality best because it reduced the influence of extreme BMI cholesterol, and glucose values without dropping patient records.",
        "The cleaned dataset has no missing values, controlled extreme values, consistent categorical fields, and is ready for downstream analysis or machine-learning tasks."
    ]
})

display(final_report)

,question,answer
0,Which imputation strategy was most effective?,MICE was the most effective multivariate strat...
1,Which outlier handling method preserved data q...,Winsorization preserved data quality best beca...
2,How did data cleaning improve dataset usability?,"The cleaned dataset has no missing values, con..."
